In [1]:
#諸々のインポート
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn import svm
from sklearn import ensemble, tree

In [2]:
def main():
    for i in range (1,101):
        addressList = pd.read_csv("data/address/delay/addressList/addressList"+str(i)+".csv", sep=",",usecols=[1,2,3])
        linerRegression(addressList,i)
        svr(addressList,i)
        bagging(addressList,i)

In [3]:
#線形回帰
def linerRegression(addressList,i):
    clf = LinearRegression(fit_intercept = True, copy_X = True, n_jobs = -1)
    for I in range(1,21):
        for line in range(len(addressList)):
            regression(addressList,addressList.address[line],addressList.lTime[line],I,clf,"linerRegression/",i)

In [4]:
#SVR(SVMを回帰に利用したもの)
def svr(addressList,i):
    clf = svm.SVR(kernel='rbf')
    for I in range(1,21):
        for line in range(len(addressList)):
             regression(addressList,addressList.address[line],addressList.lTime[line],I,clf,"svr/",i)

In [5]:
#バギング ランダムフォレスト使うやつ　アンサンブル学習
def bagging(addressList,i):
    clf = ensemble.BaggingRegressor(tree.DecisionTreeRegressor(),n_jobs = -1)
    #clf = ensemble.BaggingRegressor(tree.DecisionTreeRegressor(), n_estimators=100, max_samples=0.3)
    for I in range(1,21):
        for line in range(len(addressList)):
            regression(addressList,addressList.address[line],addressList.lTime[line],I,clf,"bagging/",i)

In [6]:
def regression(addressList,address,lTime,I,clf,regression,i):
    x_train = []
    y_train = []
    lAddress = pd.read_csv("data/address/delay/lAddress/"+address+"_"+str(i)+".csv", sep=",")
    for line in range(len(lAddress)):
        time = lAddress.time[line]
        if(lTime-time <= I):
            x_train.append(time)
            y_train.append(lAddress.rssi[line])
    clf.fit(pd.DataFrame(x_train),y_train)
    for  tmp_address in range(len(addressList)):
        #print(addressList.address[tmp_address])
        sub = addressList.fTime[tmp_address] - lTime
        if(0<=sub and sub<=6):
            x_test = []
            fAddress = pd.read_csv("data/address/delay/fAddress/"+addressList.address[tmp_address]+"_"+str(i)+".csv", sep=",")
            fTime = addressList.fTime[tmp_address]
            for time in fAddress.time:
                if(time-fTime>=0):
                    x_test.append(time)        
    predict = clf.predict(pd.DataFrame(x_test.sort()))
    write(address,I,x_test,predict,regression,i)

In [7]:
def write(address,I,data,predict,regression,i):
    f = open("data/address/delay/regression/"+regression+address+"_"+str(I)+"_"+str(i)+".csv", 'w')
    for line in range(len(predict)):
        if(line != 0):
            f.write("\r\n")
        f.write(str(data[line])+","+str(predict[line]))
    f.close()

In [8]:
main()

KeyboardInterrupt: 